# Phase 2 — Duplicate-Aware Split and ATE/ASC Export

This notebook performs the controlled splitting and task-specific export of the finance ABSA dataset.

Goals of this phase:

1. load the canonical Phase 1 dataset;
2. isolate unresolved review cases from the model-ready pool;
3. perform duplicate-aware, headline-level splitting using `group_key`;
4. apply a fixed, reproducible split policy;
5. export split-specific canonical files;
6. generate ATE-ready BIO supervision files;
7. generate ASC-ready sentence-aspect classification files;
8. verify that there is no split leakage and that exported files are internally consistent.

This phase does NOT train models yet.
It prepares the final shared dataset interface for Model A (ATE), Model B (ASC), and later pipeline/joint integration.

In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

from sklearn.model_selection import train_test_split

In [2]:
IN_CANONICAL_JSONL = Path("outputs_phase1/finance_absa_phase1_canonical.jsonl")
IN_REVIEW_QUEUE = Path("outputs_phase1/span_review_queue.csv")

OUT_DIR = Path("outputs_phase2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFEST_PATH = OUT_DIR / "split_manifest.json"
SPLIT_STATS_PATH = OUT_DIR / "split_stats_report.json"
PHASE2_CHECKLIST_PATH = OUT_DIR / "phase2_checklist.csv"

CANONICAL_TRAIN_JSONL = OUT_DIR / "train_canonical.jsonl"
CANONICAL_VAL_JSONL   = OUT_DIR / "val_canonical.jsonl"
CANONICAL_TEST_JSONL  = OUT_DIR / "test_canonical.jsonl"
CANONICAL_REVIEW_JSONL = OUT_DIR / "review_holdout_canonical.jsonl"

ATE_TRAIN_JSONL = OUT_DIR / "ate_train.jsonl"
ATE_VAL_JSONL   = OUT_DIR / "ate_val.jsonl"
ATE_TEST_JSONL  = OUT_DIR / "ate_test.jsonl"

ASC_TRAIN_JSONL = OUT_DIR / "asc_train.jsonl"
ASC_VAL_JSONL   = OUT_DIR / "asc_val.jsonl"
ASC_TEST_JSONL  = OUT_DIR / "asc_test.jsonl"

REVIEW_SUMMARY_PATH = OUT_DIR / "review_holdout_summary.json"
LEAKAGE_REPORT_PATH = OUT_DIR / "leakage_report.json"
ATE_VALIDATION_PATH = OUT_DIR / "ate_validation_report.json"
ASC_VALIDATION_PATH = OUT_DIR / "asc_validation_report.json"

## Step 1. Load the Phase 1 canonical dataset

We use the JSONL canonical file exported in Phase 1 as the sole source of truth for splitting and task-specific export.

In [3]:
records = []
with open(IN_CANONICAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

print("Canonical headline-level rows:", df.shape)
df.head(3)

Canonical headline-level rows: (10753, 7)


,raw_id,doc_id,text,group_key,num_aspects,label_signature,aspects
0,1,sentfin_000001,SpiceJet to issue 6.4 crore warrants to promoters,spicejet to issue 64 crore warrants to promoters,1,neutral,"[{'term': 'SpiceJet', 'sentiment': 'neutral', ..."
1,2,sentfin_000002,MMTC Q2 net loss at Rs 10.4 crore,mmtc q2 net loss at rs 104 crore,1,neutral,"[{'term': 'MMTC', 'sentiment': 'neutral', 'cha..."
2,3,sentfin_000003,"Mid-cap funds can deliver more, stay put: Experts",midcap funds can deliver more stay put experts,1,positive,"[{'term': 'Mid-cap funds', 'sentiment': 'posit..."


In [4]:
if IN_REVIEW_QUEUE.exists():
    df_review_queue = pd.read_csv(IN_REVIEW_QUEUE)
    print("Review queue rows:", len(df_review_queue))
else:
    df_review_queue = pd.DataFrame()
    print("No review queue file found.")

Review queue rows: 27


## Step 2. Inspect the canonical headline-level structure

Each row should represent one headline, with a list of aspect objects under the `aspects` field.
We now summarize how many headlines are fully matched versus partially unresolved.

In [5]:
def count_matched_aspects(aspects):
    return sum(1 for a in aspects if a.get("match_status") == "matched")

def count_review_aspects(aspects):
    return sum(1 for a in aspects if a.get("match_status") != "matched")

df["num_matched_aspects"] = df["aspects"].apply(count_matched_aspects)
df["num_review_aspects"] = df["aspects"].apply(count_review_aspects)
df["all_aspects_matched"] = df["num_review_aspects"] == 0

summary = {
    "headline_rows": int(len(df)),
    "all_aspects_matched_rows": int(df["all_aspects_matched"].sum()),
    "rows_with_any_review_aspect": int((~df["all_aspects_matched"]).sum()),
    "total_review_aspects": int(df["num_review_aspects"].sum()),
}
pprint(summary)

{'all_aspects_matched_rows': 10726,
 'headline_rows': 10753,
 'rows_with_any_review_aspect': 27,
 'total_review_aspects': 27}


## Step 3. Define split eligibility policy

Policy used in this project:

- Headlines with **all aspects matched** are eligible for train/val/test splitting.
- Headlines containing **any unresolved review aspect** are excluded from model-ready splits and moved to a separate review holdout.

Rationale:
For ATE, missing aspect spans would corrupt BIO supervision.
For ASC, unresolved grounding would weaken sentence-aspect consistency.
Since the unresolved pool is very small, the safest strategy is to isolate it completely.

In [6]:
df_model_ready = df[df["all_aspects_matched"]].copy().reset_index(drop=True)
df_review_holdout = df[~df["all_aspects_matched"]].copy().reset_index(drop=True)

print("Model-ready headline rows:", len(df_model_ready))
print("Review-holdout headline rows:", len(df_review_holdout))

Model-ready headline rows: 10726
Review-holdout headline rows: 27


In [7]:
with open(CANONICAL_REVIEW_JSONL, "w", encoding="utf-8") as f:
    for rec in df_review_holdout.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

review_summary = {
    "review_holdout_headlines": int(len(df_review_holdout)),
    "review_holdout_aspects": int(df_review_holdout["num_review_aspects"].sum()) if len(df_review_holdout) > 0 else 0,
}

with open(REVIEW_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(review_summary, f, ensure_ascii=False, indent=2)

print("Saved review-holdout canonical file:", CANONICAL_REVIEW_JSONL)
print("Saved review summary:", REVIEW_SUMMARY_PATH)
pprint(review_summary)

Saved review-holdout canonical file: outputs_phase2\review_holdout_canonical.jsonl
Saved review summary: outputs_phase2\review_holdout_summary.json
{'review_holdout_aspects': 27, 'review_holdout_headlines': 27}


## Step 4. Build a group-level split table

We split by `group_key`, not by individual headline row.

This prevents duplicate or near-duplicate headlines from being placed into different splits.

In [8]:
group_rows = []

for group_key, group in df_model_ready.groupby("group_key"):
    sig_counts = group["label_signature"].value_counts()
    dominant_signature = sig_counts.idxmax()
    group_rows.append({
        "group_key": group_key,
        "group_size": len(group),
        "dominant_label_signature": dominant_signature,
        "num_unique_signatures_in_group": group["label_signature"].nunique(),
        "row_ids": list(group["raw_id"])
    })

df_groups = pd.DataFrame(group_rows)
print("Group-level rows:", len(df_groups))
df_groups.head()

Group-level rows: 10658


,group_key,group_size,dominant_label_signature,num_unique_signatures_in_group,row_ids
0,10 takeaways from the secondquarter results of...,1,neutral,1,[5672]
1,10 year bond yield drops to 15 month low ahead...,1,positive,1,[4530]
2,100 days of narendra modi government sensex ra...,1,positive,1,[8265]
3,100 days of vishal sikka infosys up at 1612 pe...,1,positive,1,[8016]
4,100 infosys shares for 95k in 1993 now worth 4...,1,neutral,1,[6440]


In [9]:
mixed_groups = df_groups[df_groups["num_unique_signatures_in_group"] > 1].copy()
print("Groups with mixed label signatures:", len(mixed_groups))
mixed_groups.head(10)

Groups with mixed label signatures: 10


,group_key,group_size,dominant_label_signature,num_unique_signatures_in_group,row_ids
617,asian shares flat as investors digest greek deal,2,neutral,2,"[5695, 8950]"
1584,carl icahn says his firm sold remainder of net...,2,neutral,2,"[6246, 9400]"
2861,empee distilleries reports q3 net loss at rs 2...,2,neutral,2,"[3960, 9428]"
3367,flat nifty oi data hints at trend reversal,2,neutral,2,"[5764, 9016]"
4366,huge activity in nifty december series surpris...,2,positive,2,"[1612, 7281]"
5280,kenneth andrade exits idfc mutual fund leaving...,2,neutral,2,"[6076, 9256]"
5971,midcaps important part of our portfolio kunj b...,2,neutral+positive,2,"[2626, 8152]"
6520,nifty50 in a notrade zone due to holidaytrunca...,2,neutral,2,"[1147, 4262]"
7453,psus languishing in valuations but offer pocke...,2,negative,2,"[3815, 6139]"
10492,with investors steering clear infra bonds by b...,2,negative,2,"[4528, 6764]"


## Step 5. Define the split policy

Default split policy used here:

- train: 80%
- val: 10%
- test: 10%

The split is:
- group-aware,
- reproducible,
- stratified by dominant headline-level `label_signature`.

In [10]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

## Step 6. Perform group-aware stratified split

We first split group rows into train and temp,
then split temp into val and test using the same fixed seed.

In [11]:
df_groups_train, df_groups_temp = train_test_split(
    df_groups,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=RANDOM_SEED,
    stratify=df_groups["dominant_label_signature"]
)

temp_ratio = VAL_RATIO + TEST_RATIO
val_relative_ratio = VAL_RATIO / temp_ratio

df_groups_val, df_groups_test = train_test_split(
    df_groups_temp,
    test_size=(TEST_RATIO / temp_ratio),
    random_state=RANDOM_SEED,
    stratify=df_groups_temp["dominant_label_signature"]
)

print("Train groups:", len(df_groups_train))
print("Val groups:  ", len(df_groups_val))
print("Test groups: ", len(df_groups_test))

Train groups: 8526
Val groups:   1066
Test groups:  1066


## Step 7. Assign split labels back to headline-level rows

In [12]:
group_to_split = {}

for g in df_groups_train["group_key"]:
    group_to_split[g] = "train"
for g in df_groups_val["group_key"]:
    group_to_split[g] = "val"
for g in df_groups_test["group_key"]:
    group_to_split[g] = "test"

df_model_ready["split"] = df_model_ready["group_key"].map(group_to_split)

print(df_model_ready["split"].value_counts())

split
train    8576
test     1079
val      1071
Name: count, dtype: int64


In [13]:
assert df_model_ready["split"].isna().sum() == 0, "Some model-ready rows were not assigned a split."

## Step 8. Validate split integrity

We verify:
1. no `group_key` appears in more than one split;
2. no `raw_id` appears in more than one split;
3. split assignment is complete.

In [14]:
split_to_groups = {
    "train": set(df_model_ready.loc[df_model_ready["split"] == "train", "group_key"]),
    "val":   set(df_model_ready.loc[df_model_ready["split"] == "val", "group_key"]),
    "test":  set(df_model_ready.loc[df_model_ready["split"] == "test", "group_key"]),
}

leakage_report = {
    "train_val_overlap": len(split_to_groups["train"] & split_to_groups["val"]),
    "train_test_overlap": len(split_to_groups["train"] & split_to_groups["test"]),
    "val_test_overlap": len(split_to_groups["val"] & split_to_groups["test"]),
}

pprint(leakage_report)

{'train_test_overlap': 0, 'train_val_overlap': 0, 'val_test_overlap': 0}


In [15]:
split_to_raw_ids = {
    "train": set(df_model_ready.loc[df_model_ready["split"] == "train", "raw_id"]),
    "val":   set(df_model_ready.loc[df_model_ready["split"] == "val", "raw_id"]),
    "test":  set(df_model_ready.loc[df_model_ready["split"] == "test", "raw_id"]),
}

assert len(split_to_raw_ids["train"] & split_to_raw_ids["val"]) == 0
assert len(split_to_raw_ids["train"] & split_to_raw_ids["test"]) == 0
assert len(split_to_raw_ids["val"] & split_to_raw_ids["test"]) == 0

In [16]:
with open(LEAKAGE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(leakage_report, f, ensure_ascii=False, indent=2)

print("Saved leakage report:", LEAKAGE_REPORT_PATH)

Saved leakage report: outputs_phase2\leakage_report.json


## Step 9. Validate label-signature distribution across splits

This is not expected to be perfectly identical, but the overall distribution should remain broadly similar.

In [17]:
overall_sig = df_model_ready["label_signature"].value_counts(normalize=True).sort_index()
train_sig = df_model_ready.loc[df_model_ready["split"] == "train", "label_signature"].value_counts(normalize=True).sort_index()
val_sig = df_model_ready.loc[df_model_ready["split"] == "val", "label_signature"].value_counts(normalize=True).sort_index()
test_sig = df_model_ready.loc[df_model_ready["split"] == "test", "label_signature"].value_counts(normalize=True).sort_index()

dist_df = pd.concat(
    [overall_sig.rename("overall"), train_sig.rename("train"), val_sig.rename("val"), test_sig.rename("test")],
    axis=1
).fillna(0)

dist_df

,overall,train,val,test
label_signature,,,,
negative,0.251911,0.252215,0.251167,0.250232
negative+neutral,0.039903,0.039762,0.039216,0.041705
negative+neutral+positive,0.001119,0.001166,0.000934,0.000927
negative+positive,0.013425,0.013526,0.013072,0.012975
neutral,0.320343,0.320196,0.320261,0.321594
neutral+positive,0.060041,0.059935,0.061625,0.059314
positive,0.313258,0.313200,0.313725,0.313253


In [18]:
split_stats = {
    "num_model_ready_rows": int(len(df_model_ready)),
    "num_train_rows": int((df_model_ready["split"] == "train").sum()),
    "num_val_rows": int((df_model_ready["split"] == "val").sum()),
    "num_test_rows": int((df_model_ready["split"] == "test").sum()),
    "num_train_groups": int(len(df_groups_train)),
    "num_val_groups": int(len(df_groups_val)),
    "num_test_groups": int(len(df_groups_test)),
    "review_holdout_rows": int(len(df_review_holdout)),
}
pprint(split_stats)

{'num_model_ready_rows': 10726,
 'num_test_groups': 1066,
 'num_test_rows': 1079,
 'num_train_groups': 8526,
 'num_train_rows': 8576,
 'num_val_groups': 1066,
 'num_val_rows': 1071,
 'review_holdout_rows': 27}


## Step 10. Export split-specific canonical files

These files remain headline-level and preserve the full list of matched aspects for each headline.
They are the shared interface for all downstream task-specific exports.

In [19]:
def save_jsonl_from_df(df_in, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in df_in.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "train"], CANONICAL_TRAIN_JSONL)
save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "val"],   CANONICAL_VAL_JSONL)
save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "test"],  CANONICAL_TEST_JSONL)

print("Saved:", CANONICAL_TRAIN_JSONL)
print("Saved:", CANONICAL_VAL_JSONL)
print("Saved:", CANONICAL_TEST_JSONL)

Saved: outputs_phase2\train_canonical.jsonl
Saved: outputs_phase2\val_canonical.jsonl
Saved: outputs_phase2\test_canonical.jsonl


## Step 11. Build ATE-ready word-level supervision

ATE requires BIO sequence labeling.
We therefore convert each matched headline into:
- text
- token list
- token character offsets
- BIO labels
- aspect span metadata

Later, the model engineer can map these word-level BIO labels into DeBERTa subword labels.

In [20]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")

def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_PATTERN.finditer(text)]

In [21]:
def build_bio_labels(tokens_with_offsets, aspects):
    labels = ["O"] * len(tokens_with_offsets)
    
    # only matched aspects
    matched_aspects = [a for a in aspects if a.get("match_status") == "matched"]
    matched_aspects = sorted(matched_aspects, key=lambda x: (x["char_from"], x["char_to"]))
    
    for asp in matched_aspects:
        a_from = asp["char_from"]
        a_to = asp["char_to"]
        
        token_indices = []
        for i, (_, t_from, t_to) in enumerate(tokens_with_offsets):
            # token fully inside aspect span
            if t_from >= a_from and t_to <= a_to:
                token_indices.append(i)
        
        if len(token_indices) == 0:
            continue
        
        labels[token_indices[0]] = "B-ASP"
        for j in token_indices[1:]:
            labels[j] = "I-ASP"
    
    return labels

In [22]:
def make_ate_records(df_split):
    out = []
    for _, row in df_split.iterrows():
        text = row["text"]
        aspects = row["aspects"]
        toks = tokenize_with_offsets(text)
        bio = build_bio_labels(toks, aspects)
        
        out.append({
            "doc_id": row["doc_id"],
            "raw_id": row["raw_id"],
            "split": row["split"],
            "text": text,
            "tokens": [t[0] for t in toks],
            "token_offsets": [(t[1], t[2]) for t in toks],
            "bio_labels": bio,
            "aspects": aspects
        })
    return out

## Step 12. Validate ATE supervision

We verify:
1. token count equals BIO label count;
2. no split leakage in exported ATE records;
3. BIO spans can be approximately recovered back into aspect surface forms.

In [23]:
ate_train = make_ate_records(df_model_ready[df_model_ready["split"] == "train"])
ate_val   = make_ate_records(df_model_ready[df_model_ready["split"] == "val"])
ate_test  = make_ate_records(df_model_ready[df_model_ready["split"] == "test"])

print(len(ate_train), len(ate_val), len(ate_test))

8576 1071 1079


In [24]:
def recover_bio_spans(tokens, labels):
    spans = []
    current = []
    for tok, lab in zip(tokens, labels):
        if lab == "B-ASP":
            if current:
                spans.append(" ".join(current))
            current = [tok]
        elif lab == "I-ASP":
            if current:
                current.append(tok)
        else:
            if current:
                spans.append(" ".join(current))
                current = []
    if current:
        spans.append(" ".join(current))
    return spans

ate_validation = {
    "train_token_label_len_ok": all(len(r["tokens"]) == len(r["bio_labels"]) for r in ate_train),
    "val_token_label_len_ok": all(len(r["tokens"]) == len(r["bio_labels"]) for r in ate_val),
    "test_token_label_len_ok": all(len(r["tokens"]) == len(r["bio_labels"]) for r in ate_test),
    "train_rows": len(ate_train),
    "val_rows": len(ate_val),
    "test_rows": len(ate_test),
}

pprint(ate_validation)

{'test_rows': 1079,
 'test_token_label_len_ok': True,
 'train_rows': 8576,
 'train_token_label_len_ok': True,
 'val_rows': 1071,
 'val_token_label_len_ok': True}


In [25]:
with open(ATE_VALIDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(ate_validation, f, ensure_ascii=False, indent=2)

print("Saved:", ATE_VALIDATION_PATH)

Saved: outputs_phase2\ate_validation_report.json


In [26]:
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(ate_train, ATE_TRAIN_JSONL)
save_jsonl(ate_val, ATE_VAL_JSONL)
save_jsonl(ate_test, ATE_TEST_JSONL)

print("Saved:", ATE_TRAIN_JSONL)
print("Saved:", ATE_VAL_JSONL)
print("Saved:", ATE_TEST_JSONL)

Saved: outputs_phase2\ate_train.jsonl
Saved: outputs_phase2\ate_val.jsonl
Saved: outputs_phase2\ate_test.jsonl


## Step 13. Build ASC-ready aspect-level classification records

ASC requires one record per (sentence, aspect, sentiment) pair.
Only matched aspects are exported.

In [27]:
def make_asc_records(df_split):
    out = []
    for _, row in df_split.iterrows():
        for asp in row["aspects"]:
            if asp.get("match_status") != "matched":
                continue
            out.append({
                "doc_id": row["doc_id"],
                "raw_id": row["raw_id"],
                "split": row["split"],
                "sentence": row["text"],
                "aspect": asp["term"],
                "sentiment": asp["sentiment"],
                "char_from": asp["char_from"],
                "char_to": asp["char_to"],
                "match_type": asp["match_type"],
                "input_text_template": f"{row['text']} [SEP] {asp['term']}"
            })
    return out

In [28]:
asc_train = make_asc_records(df_model_ready[df_model_ready["split"] == "train"])
asc_val   = make_asc_records(df_model_ready[df_model_ready["split"] == "val"])
asc_test  = make_asc_records(df_model_ready[df_model_ready["split"] == "test"])

print(len(asc_train), len(asc_val), len(asc_test))

11443 1438 1468


## Step 14. Validate ASC supervision

We verify:
1. no unresolved review aspects appear in ASC files;
2. all labels belong to the allowed 3-class set;
3. split counts are non-empty and consistent.

In [29]:
ALLOWED_SENTIMENTS = {"positive", "negative", "neutral"}

def asc_valid(records):
    return all(r["sentiment"] in ALLOWED_SENTIMENTS for r in records)

asc_validation = {
    "train_all_labels_valid": asc_valid(asc_train),
    "val_all_labels_valid": asc_valid(asc_val),
    "test_all_labels_valid": asc_valid(asc_test),
    "train_records": len(asc_train),
    "val_records": len(asc_val),
    "test_records": len(asc_test),
}

pprint(asc_validation)

{'test_all_labels_valid': True,
 'test_records': 1468,
 'train_all_labels_valid': True,
 'train_records': 11443,
 'val_all_labels_valid': True,
 'val_records': 1438}


In [30]:
with open(ASC_VALIDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(asc_validation, f, ensure_ascii=False, indent=2)

print("Saved:", ASC_VALIDATION_PATH)

Saved: outputs_phase2\asc_validation_report.json


In [31]:
save_jsonl(asc_train, ASC_TRAIN_JSONL)
save_jsonl(asc_val, ASC_VAL_JSONL)
save_jsonl(asc_test, ASC_TEST_JSONL)

print("Saved:", ASC_TRAIN_JSONL)
print("Saved:", ASC_VAL_JSONL)
print("Saved:", ASC_TEST_JSONL)

Saved: outputs_phase2\asc_train.jsonl
Saved: outputs_phase2\asc_val.jsonl
Saved: outputs_phase2\asc_test.jsonl


## Step 15. Save the split manifest

This file records the exact split policy and key metadata required for reproducibility.

In [32]:
split_manifest = {
    "random_seed": RANDOM_SEED,
    "split_policy": {
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO
    },
    "split_unit": "group_key",
    "stratify_field": "dominant_label_signature",
    "review_policy": "headlines with any unresolved aspect are excluded to review holdout",
    "train_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "train", "doc_id"]),
    "val_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "val", "doc_id"]),
    "test_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "test", "doc_id"]),
}

with open(SPLIT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)

print("Saved split manifest:", SPLIT_MANIFEST_PATH)

Saved split manifest: outputs_phase2\split_manifest.json


## Step 16. Save split statistics report

In [33]:
split_stats_report = {
    **split_stats,
    "leakage_report": leakage_report,
    "review_summary": review_summary,
    "ate_validation": ate_validation,
    "asc_validation": asc_validation
}

with open(SPLIT_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(split_stats_report, f, ensure_ascii=False, indent=2)

print("Saved split stats report:", SPLIT_STATS_PATH)
pprint(split_stats_report)

Saved split stats report: outputs_phase2\split_stats_report.json
{'asc_validation': {'test_all_labels_valid': True,
                    'test_records': 1468,
                    'train_all_labels_valid': True,
                    'train_records': 11443,
                    'val_all_labels_valid': True,
                    'val_records': 1438},
 'ate_validation': {'test_rows': 1079,
                    'test_token_label_len_ok': True,
                    'train_rows': 8576,
                    'train_token_label_len_ok': True,
                    'val_rows': 1071,
                    'val_token_label_len_ok': True},
 'leakage_report': {'train_test_overlap': 0,
                    'train_val_overlap': 0,
                    'val_test_overlap': 0},
 'num_model_ready_rows': 10726,
 'num_test_groups': 1066,
 'num_test_rows': 1079,
 'num_train_groups': 8526,
 'num_train_rows': 8576,
 'num_val_groups': 1066,
 'num_val_rows': 1071,
 'review_holdout_rows': 27,
 'review_summary': {'review_holdou

## Step 17. Evaluate whether Phase 2 is complete

Phase 2 is complete if:
- the model-ready pool is separated from the review holdout;
- duplicate-aware group splitting is performed;
- no group leakage exists across train/val/test;
- canonical split files are exported;
- ATE files are exported and BIO labels are well-formed;
- ASC files are exported and labels are valid;
- a split manifest is saved for reproducibility.

In [34]:
checklist = [
    ("Canonical input loaded", len(df) > 0),
    ("Review holdout created", CANONICAL_REVIEW_JSONL.exists()),
    ("All model-ready rows assigned split", df_model_ready["split"].isna().sum() == 0),
    ("No train/val group leakage", leakage_report["train_val_overlap"] == 0),
    ("No train/test group leakage", leakage_report["train_test_overlap"] == 0),
    ("No val/test group leakage", leakage_report["val_test_overlap"] == 0),
    ("Train canonical exported", CANONICAL_TRAIN_JSONL.exists()),
    ("Val canonical exported", CANONICAL_VAL_JSONL.exists()),
    ("Test canonical exported", CANONICAL_TEST_JSONL.exists()),
    ("ATE train exported", ATE_TRAIN_JSONL.exists()),
    ("ATE val exported", ATE_VAL_JSONL.exists()),
    ("ATE test exported", ATE_TEST_JSONL.exists()),
    ("ASC train exported", ASC_TRAIN_JSONL.exists()),
    ("ASC val exported", ASC_VAL_JSONL.exists()),
    ("ASC test exported", ASC_TEST_JSONL.exists()),
    ("ATE validation passed", ate_validation["train_token_label_len_ok"] and ate_validation["val_token_label_len_ok"] and ate_validation["test_token_label_len_ok"]),
    ("ASC validation passed", asc_validation["train_all_labels_valid"] and asc_validation["val_all_labels_valid"] and asc_validation["test_all_labels_valid"]),
    ("Split manifest exported", SPLIT_MANIFEST_PATH.exists()),
]

df_checklist = pd.DataFrame(checklist, columns=["check_item", "passed"])
df_checklist.to_csv(PHASE2_CHECKLIST_PATH, index=False)

display(df_checklist)

all_passed = df_checklist["passed"].all()
print("PHASE 2 COMPLETE:", all_passed)

,check_item,passed
0,Canonical input loaded,True
1,Review holdout created,True
2,All model-ready rows assigned split,True
3,No train/val group leakage,True
4,No train/test group leakage,True
5,No val/test group leakage,True
6,Train canonical exported,True
7,Val canonical exported,True
8,Test canonical exported,True
9,ATE train exported,True


PHASE 2 COMPLETE: True


## Phase 2 conclusion

This phase is considered successful if the canonical finance ABSA dataset has been converted into:

1. a duplicate-aware, reproducible train/val/test split;
2. split-specific canonical files;
3. ATE-ready word-level BIO supervision files;
4. ASC-ready aspect-level classification files;
5. a review holdout for unresolved aspect-grounding cases.

The next phase will focus on model-facing preprocessing:

- tokenizer-specific conversion for DeBERTa,
- subword alignment for BIO labels,
- PyTorch/HuggingFace Dataset and DataLoader construction,
- and shared dataset interfaces for Member 3 (ATE), Member 4 (ASC), and Member 5 (joint model).